<a href="https://colab.research.google.com/github/manasvimittal27/CareerCopilot-AI/blob/main/Masters_Union_Class1_Life_Inbox_Colab_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Term 1 Outclass 1: From Software to Intelligent Software

**Class goal:** build the first tiny slice of a real product and understand the request flow behind it.

Today we will move through three ideas:

1. How normal software works: client -> API -> backend -> database.
2. How AI helps us build software faster.
3. How a product itself can call an LLM and become more intelligent.

**Reference product:** Life Inbox, a personal organizer for messy notes, tasks, links and ideas.


## Mental Model


The most important mental model for today:

```text
Browser / client -> HTTP request -> Python backend -> database -> response -> browser
```

Then add intelligence without changing that mental model:

```text
Browser / client -> Python backend -> database
                              |
                              v
                           OpenAI API
```

For the AI demo, analyze the **whole saved inbox at once**. The point is to show that an LLM can take a pile of messy, unstructured notes and turn it into useful product structure: buckets, priorities, and concrete ToDos.


## 0. Setup

Run this first. In Colab, package installation can take a minute.


In [ ]:
!pip -q install fastapi uvicorn nest_asyncio pydantic openai requests


## 1. API key setup

For the AI demo, you need an OpenAI API key.

Recommended for class:
- Keep the key private.
- Do not paste it in chat.
- Do not put it inside frontend/browser JavaScript.
- Backend code reads it from the environment.

This notebook will still work without a key because it includes a small fallback organizer. The fallback is useful if internet/API setup misbehaves during class.


In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    key = getpass("Paste your OpenAI API key. Leave blank to use fallback mode: ")
    if key.strip():
        os.environ["OPENAI_API_KEY"] = key.strip()

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))


OPENAI_API_KEY set: True


## 2. Start with a product idea

The product is intentionally simple at first.

**Life Inbox v0**

A user can:
- type a note,
- save it,
- see it later.

This is enough to teach clients, APIs, servers and databases.


In [ ]:
# A tiny in-memory version before we use a database.
# This is not a web app yet. It is just product logic in Python.

notes = []

def add_note(text):
    note = {
        "id": len(notes) + 1,
        "text": text,
    }
    notes.append(note)
    return note

def list_notes():
    return notes

add_note("Call CA tomorrow about tax filing")
add_note("Idea: build a life inbox that organizes random thoughts")
list_notes()


[{'id': 1, 'text': 'Call CA tomorrow about tax filing'},
 {'id': 2, 'text': 'Idea: build a life inbox that organizes random thoughts'}]

### Persistence or the lack thereof


- If I restart this notebook, where did the notes go?
- What do we need if the product should remember things tomorrow?



## 3. Add a database

We will use SQLite because it is built into Python and keeps the setup simple.

In a production system, this may later become Postgres, MySQL, DynamoDB, etc. The mental model is the same: persistent state lives outside the transient request handler.

**Classroom convenience:** the next cell includes `RESET_DB = False`. Flip it to `True` and run the cell once whenever you want a completely clean demo state.


In [ ]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

DB_PATH = "life_inbox.db"

# ------------------------------------------------------------------
# OPTIONAL CLASSROOM RESET
# Flip this to True whenever you want to start the demo from a clean DB.
# IMPORTANT: set it back to False after running once, otherwise every
# rerun of this cell will wipe the database again.
# ------------------------------------------------------------------
RESET_DB = True

if RESET_DB and Path(DB_PATH).exists():
    Path(DB_PATH).unlink()
    print("Deleted existing database. Starting clean.")

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
        """)
        conn.commit()

init_db()
print("Database ready:", Path(DB_PATH).resolve())


Deleted existing database. Starting clean.
Database ready: /content/life_inbox.db


In [ ]:
def db_add_note(text):
    now = datetime.now(timezone.utc).isoformat()
    with get_conn() as conn:
        cur = conn.execute(
            "INSERT INTO notes (text, created_at) VALUES (?, ?)",
            (text, now),
        )
        conn.commit()
        note_id = cur.lastrowid
    return {"id": note_id, "text": text, "created_at": now}


def db_list_notes():
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT id, text, created_at FROM notes ORDER BY id DESC"
        ).fetchall()
    return [dict(row) for row in rows]


#db_add_note("We are building a chatbot.")
#db_add_note("Need to revise whatever I learnt.")

db_list_notes()


[{'id': 28,
  'text': 'Prototype mobile payment flow that works offline (sketch idea).',
  'created_at': '2026-09-08T13:10:27.640073+00:00'},
 {'id': 27,
  'text': 'Receipt shows refunded parking charge, check bank Monday.',
  'created_at': '2026-09-08T13:10:27.631374+00:00'},
 {'id': 26,
  'text': 'Shop for ergonomic keyboard and test wrist angle at office.',
  'created_at': '2026-09-08T13:10:27.622405+00:00'},
 {'id': 25,
  'text': 'Ask HR about parental leave policy before next paycheck cycle.',
  'created_at': '2026-09-08T13:10:27.613587+00:00'},
 {'id': 24,
  'text': 'Try Duolingo Spanish lessons for 15 minutes every morning.',
  'created_at': '2026-09-08T13:10:27.604349+00:00'},
 {'id': 23,
  'text': 'Store receipts for last conference for expense report.',
  'created_at': '2026-09-08T13:10:27.594405+00:00'},
 {'id': 22,
  'text': 'Pick up dry cleaning on way home.',
  'created_at': '2026-09-08T13:10:27.584089+00:00'},
 {'id': 21,
  'text': 'Book quick flight red-eye to SFO if ch

### Optional: generate a realistic inbox with 20–30 notes

For a richer classroom demo, we can ask an LLM to generate a messy but realistic personal inbox and save all of those notes into SQLite.

Keep `GENERATE_DEMO_NOTES = False` during normal runs. Flip it to `True` only when you want to populate the database.

A local fallback list is included so the class can continue even if the API is unavailable.


In [ ]:
import json
import os

GENERATE_DEMO_NOTES = True
NUM_DEMO_NOTES = 25
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")

FALLBACK_DEMO_NOTES = [
    "Call CA tomorrow about the advance tax calculation.",
    "Idea: build a single inbox that organizes notes, reminders, links and random thoughts.",
    "Buy a birthday gift for Mom before the weekend.",
    "Rohan recommended Atomic Habits. Order it this weekend.",
    "Ask Rohan for the name of that Gurgaon restaurant he mentioned.",
    "Finish the architecture review document before Thursday afternoon.",
    "Interesting article on vector databases. Read it when I get time.",
    "Electricity bill is due on the 12th. Pay before the due date.",
    "Book dentist appointment for next week.",
    "Potential app idea: voice notes that automatically become tasks and reminders.",
    "Follow up with recruiter if there is no response by Friday.",
    "Need to compare health insurance renewal options for parents.",
    "Pick up dry cleaning on the way home.",
    "Watch the system design lecture on caching and take notes.",
    "Remember to send the reimbursement documents to finance.",
    "Trip idea: 3-day family break in Jaipur during the next long weekend.",
    "Research whether the Life Inbox app should use Postgres or SQLite initially.",
    "Call Papa in the evening and ask about the property tax receipt.",
    "Need groceries: curd, milk, bread, bananas and coffee.",
    "Idea for class: demonstrate the same API from browser, Python and mobile client.",
    "Prepare questions for tomorrow's interview practice.",
    "Renew domain name before it expires later this month.",
    "Send the meeting summary to the team.",
    "Look into a lightweight workout routine that can fit into 20 minutes.",
    "Read the saved article about prompt injection attacks.",
]

def _extract_json_array(raw: str):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()
    start = raw.find("[")
    end = raw.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON array found in model output")
    value = json.loads(raw[start:end + 1])
    if not isinstance(value, list):
        raise ValueError("Expected a JSON array")
    return value

def generate_demo_notes(n=25):
    if not os.getenv("OPENAI_API_KEY"):
        print("No API key found. Using local fallback notes.")
        return FALLBACK_DEMO_NOTES[:n]

    try:
        from openai import OpenAI
        client = OpenAI()

        prompt = f"""
Generate exactly {n} realistic raw notes from one busy person's personal inbox.

The notes should feel like things someone quickly typed during normal life.
Mix:
- work follow-ups
- personal errands
- money / bills
- learning
- shopping
- people to contact
- travel ideas
- product / startup ideas
- appointments
- reminders
- a few informational notes with no action required

Requirements:
- Some notes should contain explicit urgency or timing.
- Some should imply a ToDo.
- Some should be ideas or reference material only.
- Keep each note to one short sentence.
- Do not pre-categorize, number, or explain them.
- Return ONLY a JSON array of strings.
"""

        response = client.responses.create(model=MODEL, input=prompt)
        generated = _extract_json_array(response.output_text)

        clean = [str(x).strip() for x in generated if str(x).strip()]
        if len(clean) < 20:
            raise ValueError(f"Model returned only {len(clean)} usable notes")
        return clean[:n]

    except Exception as e:
        print("LLM generation failed; using fallback notes.")
        print("Reason:", str(e)[:300])
        return FALLBACK_DEMO_NOTES[:n]

if GENERATE_DEMO_NOTES:
    generated_notes = generate_demo_notes(NUM_DEMO_NOTES)
    for note_text in generated_notes:
        db_add_note(note_text)

    print(f"Added {len(generated_notes)} demo notes to the database.")
    print(f"Database now contains {len(db_list_notes())} notes.")
else:
    print("Demo-note generation skipped. Set GENERATE_DEMO_NOTES = True to populate the DB.")


Added 25 demo notes to the database.
Database now contains 28 notes.


## 4. Turn the Python logic into a web application

Now we introduce three pieces:

```text
HTML/JS client -> FastAPI backend -> SQLite database
```

The browser does not directly talk to SQLite. It talks to our backend through API endpoints.


### Build the web app in small layers

The complete application is intentionally split into small blocks so we can connect each piece of code back to the architecture diagram.

**Run blocks 4A → 4H in order.** Block 4A creates `app.py`; every later block appends the next logical layer.


### 4A. App setup + database layer

Create `app.py`, initialize FastAPI, and define the SQLite connection/setup. This first block **overwrites** `app.py`; the blocks after it append to the same file.


In [ ]:
%%writefile app.py
import os
import json
import sqlite3
from datetime import datetime, timezone
from typing import List

from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

DB_PATH = "life_inbox.db"
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")

app = FastAPI(title="Life Inbox")


# -----------------------------
# Database layer
# -----------------------------
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
        """)
        conn.commit()


init_db()




Overwriting app.py


### 4B. Data contracts + normal product logic

Define the request/response shapes and the ordinary Python functions that save and read notes. No AI yet.


In [ ]:
%%writefile -a app.py
# -----------------------------
# Data contracts
# -----------------------------
class NoteCreate(BaseModel):
    text: str = Field(min_length=1, max_length=2000)


class Note(BaseModel):
    id: int
    text: str
    created_at: str


# -----------------------------
# Product logic
# -----------------------------
def add_note_to_db(text: str) -> dict:
    now = datetime.now(timezone.utc).isoformat()
    with get_conn() as conn:
        cur = conn.execute(
            "INSERT INTO notes (text, created_at) VALUES (?, ?)",
            (text, now),
        )
        conn.commit()
        note_id = cur.lastrowid
    return {"id": note_id, "text": text, "created_at": now}


def list_notes_from_db() -> list[dict]:
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT id, text, created_at FROM notes ORDER BY id ASC"
        ).fetchall()
    return [dict(row) for row in rows]




Appending to app.py


### 4C. AI helpers + deterministic fallback

Add JSON parsing plus a simple non-LLM fallback. This is useful both for teaching and for keeping the demo alive if the model API is unavailable.


In [ ]:
%%writefile -a app.py
# -----------------------------
# AI logic
# -----------------------------
def extract_json_object(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.replace("```json", "").replace("```", "").strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in model output")
    return json.loads(raw[start:end + 1])


def priority_for_text(text: str) -> str:
    lowered = text.lower()

    high_markers = [
        "today", "tomorrow", "urgent", "asap", "due", "before friday",
        "before thursday", "expires", "deadline"
    ]
    medium_markers = [
        "this week", "weekend", "next week", "follow up", "remember",
        "need to", "appointment", "renew"
    ]

    if any(x in lowered for x in high_markers):
        return "High"
    if any(x in lowered for x in medium_markers):
        return "Medium"
    return "Low"


def bucket_for_text(text: str) -> str:
    lowered = text.lower()

    rules = [
        ("Work", ["team", "meeting", "architecture", "recruiter", "finance", "interview", "document"]),
        ("Money & Admin", ["bill", "tax", "insurance", "domain", "property tax", "reimbursement", "ca "]),
        ("Learning", ["read", "watch", "lecture", "article", "learn", "vector database", "prompt injection"]),
        ("Shopping & Errands", ["buy", "groceries", "pick up", "order", "dry cleaning"]),
        ("Health", ["dentist", "workout", "health"]),
        ("Travel", ["trip", "travel", "hotel", "jaipur"]),
        ("Ideas", ["idea", "build", "app", "startup", "product"]),
        ("People", ["call", "ask", "rohan", "papa", "mom"]),
    ]

    for name, markers in rules:
        if any(marker in lowered for marker in markers):
            return name
    return "Personal / Reference"


def fallback_analyze(notes: list[dict]) -> dict:
    """Deterministic fallback so the demo still works if the API is unavailable."""
    buckets = {}
    todos = []
    priority_view = {"High": [], "Medium": [], "Low": []}

    action_markers = [
        "call", "buy", "send", "ask", "finish", "submit", "pay", "book",
        "pick up", "prepare", "renew", "follow up", "need to", "remember to",
        "research", "look into", "order"
    ]

    for note in notes:
        text = note["text"]
        bucket = bucket_for_text(text)
        priority = priority_for_text(text)
        title = text[:72] + ("..." if len(text) > 72 else "")

        item = {
            "id": note["id"],
            "title": title,
            "priority": priority,
        }

        buckets.setdefault(bucket, []).append(item)
        priority_view[priority].append({"id": note["id"], "title": title})

        if any(marker in text.lower() for marker in action_markers):
            todos.append({
                "task": text,
                "priority": priority,
                "source_note_ids": [note["id"]],
                "when": None,
            })

    return {
        "buckets": [
            {"name": name, "notes": items}
            for name, items in buckets.items()
        ],
        "todos": todos,
        "priority_view": priority_view,
        "mode": "fallback_no_api_call",
    }




Appending to app.py


### 4D. Whole-inbox LLM analysis

Add the function that sends **all saved notes together** to the LLM and asks for buckets, priorities, and concrete ToDos.


In [ ]:
%%writefile -a app.py
def analyze_inbox_with_openai(notes: list[dict]) -> dict:
    if not notes:
        return {
            "buckets": [],
            "todos": [],
            "priority_view": {"High": [], "Medium": [], "Low": []},
            "mode": "empty_inbox",
        }

    if not os.getenv("OPENAI_API_KEY"):
        return fallback_analyze(notes)

    try:
        from openai import OpenAI
        client = OpenAI()

        compact_notes = [{"id": n["id"], "text": n["text"]} for n in notes]

        prompt = f"""
You are the organization engine for a personal Life Inbox.

Analyze ALL saved notes together and turn the messy inbox into a practical dashboard.

Return ONLY valid JSON in exactly this shape:

{{
  "buckets": [
    {{
      "name": "short useful category name",
      "notes": [
        {{
          "id": 1,
          "title": "concise title preserving the note's meaning",
          "priority": "High"
        }}
      ]
    }}
  ],
  "todos": [
    {{
      "task": "concrete action",
      "priority": "High",
      "source_note_ids": [1],
      "when": "timing mentioned in the note, or null"
    }}
  ],
  "priority_view": {{
    "High": [
      {{"id": 1, "title": "concise title"}}
    ],
    "Medium": [],
    "Low": []
  }}
}}

Rules:
1. Organize the full inbox, not each note independently.
2. Every saved note must appear in exactly one bucket.
3. Use a small number of useful buckets such as Work, Personal, Money & Admin,
   Learning, Shopping & Errands, Health, Travel, Ideas, People, or another
   category only when it is genuinely clearer.
4. Priority must be one of High, Medium, Low.
5. High priority means explicit urgency, a near deadline, or a clearly time-sensitive obligation.
6. Medium means actionable but not immediately urgent.
7. Low means reference material, someday ideas, or non-urgent items.
8. Extract ToDos only when the note contains an explicit or strongly implied action.
9. A note can produce multiple ToDos if it genuinely contains multiple actions.
10. Do not invent dates, deadlines, people, facts, or actions.
11. Preserve relative timing exactly when present, for example "tomorrow" or "this weekend".
12. Keep titles and ToDos short and concrete. No motivational language, commentary, or prose outside the JSON.
13. priority_view should include every note once, grouped by priority.

Saved notes:
{json.dumps(compact_notes, ensure_ascii=False)}
"""

        response = client.responses.create(model=MODEL, input=prompt)
        result = extract_json_object(response.output_text)
        result["mode"] = "openai"
        return result

    except Exception as e:
        fallback = fallback_analyze(notes)
        fallback["mode"] = "fallback_after_api_error"
        fallback["error"] = str(e)[:300]
        return fallback




Appending to app.py


### 4E. API endpoints

Expose the product capabilities as HTTP endpoints: health, create note, list notes, and analyze the entire inbox.


In [ ]:
%%writefile -a app.py
# -----------------------------
# API endpoints
# -----------------------------
@app.get("/api/health")
def health():
    return {"status": "ok"}


@app.post("/api/notes", response_model=Note)
def create_note(payload: NoteCreate):
    return add_note_to_db(payload.text)


@app.get("/api/notes", response_model=List[Note])
def get_notes():
    return list_notes_from_db()


@app.post("/api/analyze")
def analyze_inbox():
    return analyze_inbox_with_openai(list_notes_from_db())




Appending to app.py


### 4F. Tiny frontend: HTML + CSS

Start the browser UI. This block contains the page structure and styling, and begins the JavaScript section.


In [ ]:
%%writefile -a app.py
# -----------------------------
# Very small frontend
# -----------------------------
@app.get("/", response_class=HTMLResponse)
def index():
    return """
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Life Inbox</title>
  <style>
    body { font-family: system-ui, -apple-system, Segoe UI, sans-serif; margin: 40px; background: #f6f8fa; color: #111; }
    .wrap { max-width: 920px; margin: 0 auto; }
    h1 { margin-bottom: 6px; }
    h3 { margin-bottom: 8px; }
    .sub { color: #555; margin-top: 0; }
    textarea { width: 100%; min-height: 100px; font-size: 16px; padding: 12px; border-radius: 10px; border: 1px solid #ccc; box-sizing: border-box; }
    button { margin-top: 12px; margin-right: 8px; padding: 10px 14px; border: 0; border-radius: 8px; cursor: pointer; background: #0b7285; color: white; font-weight: 650; }
    button.secondary { background: #495057; }
    .panel { background: white; border: 1px solid #e5e7eb; border-radius: 12px; padding: 16px; margin: 12px 0; }
    .note { background: white; border: 1px solid #e5e7eb; border-radius: 12px; padding: 14px; margin: 12px 0; box-shadow: 0 1px 2px rgba(0,0,0,0.04); }
    .time { color: #777; font-size: 12px; margin-top: 6px; }
    .pill { display: inline-block; padding: 2px 8px; border-radius: 999px; background: #eef2f7; margin-left: 6px; font-size: 12px; }
    ul { padding-left: 22px; }
    .muted { color: #666; }
  </style>
</head>
<body>
  <div class="wrap">
    <h1>Life Inbox</h1>
    <p class="sub">Capture first. Organize the whole inbox with AI when you are ready.</p>

    <textarea id="noteText" placeholder="Example: Met Rohan yesterday. He recommended Atomic Habits. Need to order it this weekend. Also ask him about that Gurgaon restaurant."></textarea>
    <br />
    <button onclick="addNote()">Add Note</button>
    <button class="secondary" onclick="analyzeInbox()">Analyze Entire Inbox with AI</button>

    <h2>AI-organized inbox</h2>
    <div id="aiOutput" class="panel">
      <span class="muted">Save a few notes, then click “Analyze Entire Inbox with AI”.</span>
    </div>

    <h2>Raw saved notes</h2>
    <div id="notes"></div>
  </div>

<script>


Appending to app.py


### 4G. Tiny frontend: capture + render analysis

Add the browser-side functions for saving notes, escaping text safely, and rendering the AI-organized dashboard.


In [ ]:
%%writefile -a app.py
async function addNote() {
  const text = document.getElementById('noteText').value.trim();
  if (!text) return alert('Write a note first');

  const response = await fetch('/api/notes', {
    method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({text})
  });
  const data = await response.json();
  console.log('Created note:', data);

  document.getElementById('noteText').value = '';
  await loadNotes();
}

function esc(str) {
  return String(str ?? '').replace(/[&<>'"]/g, tag => ({
    '&':'&amp;', '<':'&lt;', '>':'&gt;', "'":'&#39;', '"':'&quot;'
  }[tag]));
}

function renderAnalysis(data) {
  const root = document.getElementById('aiOutput');

  const bucketHtml = (data.buckets || []).map(bucket => `
    <div class="panel">
      <h3>${esc(bucket.name)}</h3>
      <ul>
        ${(bucket.notes || []).map(n =>
          `<li>${esc(n.title)} <span class="pill">${esc(n.priority)}</span></li>`
        ).join('')}
      </ul>
    </div>
  `).join('');

  const todoHtml = (data.todos || []).map(t =>
    `<li><strong>${esc(t.task)}</strong> <span class="pill">${esc(t.priority)}</span>${t.when ? ` · ${esc(t.when)}` : ''}</li>`
  ).join('');

  const pv = data.priority_view || {High: [], Medium: [], Low: []};
  const priorityHtml = ['High', 'Medium', 'Low'].map(level => `
    <div class="panel">
      <h3>${level} priority (${(pv[level] || []).length})</h3>
      <ul>
        ${(pv[level] || []).map(n => `<li>${esc(n.title)}</li>`).join('') || '<li class="muted">None</li>'}
      </ul>
    </div>
  `).join('');

  root.innerHTML = `
    <h3>Buckets</h3>
    ${bucketHtml || '<p class="muted">No notes yet.</p>'}

    <h3>ToDos</h3>
    <ul>${todoHtml || '<li class="muted">No concrete ToDos detected.</li>'}</ul>

    <h3>Priority view</h3>
    ${priorityHtml}

    <p class="muted">Mode: ${esc(data.mode || 'unknown')}</p>
  `;
}



Appending to app.py


### 4H. Tiny frontend: analyze + load notes

Finish the browser-side functions, close the page, and complete `app.py`.


In [ ]:
%%writefile -a app.py
async function analyzeInbox() {
  const root = document.getElementById('aiOutput');
  root.innerHTML = '<span class="muted">Analyzing the complete saved inbox...</span>';

  const response = await fetch('/api/analyze', { method: 'POST' });
  const data = await response.json();

  console.log('AI inbox analysis:', data);
  renderAnalysis(data);
}

async function loadNotes() {
  const response = await fetch('/api/notes');
  const notes = await response.json();
  const root = document.getElementById('notes');
  root.innerHTML = '';

  for (const note of notes) {
    const div = document.createElement('div');
    div.className = 'note';
    div.innerHTML = `<div>${esc(note.text)}</div><div class="time">#${note.id} · ${esc(note.created_at)}</div>`;
    root.appendChild(div);
  }
}

loadNotes();
</script>
</body>
</html>
"""


Appending to app.py


## 5. Run the web app

This starts the FastAPI server in the background.

In Colab, after the server starts, use the `open_app()` helper in the next cell to open it.


In [ ]:
import subprocess, time, os, signal, requests

# Stop an earlier server if this cell is rerun.
try:
    server.terminate()
    time.sleep(1)
except Exception:
    pass

server = subprocess.Popen([
    "python", "-m", "uvicorn", "app:app",
    "--host", "0.0.0.0",
    "--port", "8000",
])

time.sleep(3)
print("Server running on port 8000")
print("Health check:", requests.get("http://127.0.0.1:8000/api/health").json())


Server running on port 8000
Health check: {'status': 'ok'}


In [ ]:
def open_app(port=8000):
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(port)
    except Exception:
        print(f"Open http://127.0.0.1:{port} in your browser")

open_app(8000)


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## 6. Use Python as a client

A browser is one kind of client. Python can also be a client.

This is useful if the browser demo fails or if you want to show the API without UI distractions.


In [ ]:
import requests

base = "http://127.0.0.1:8000"

payload = {
    "text": "Met Rohan yesterday. He recommended Atomic Habits. Need to order it this weekend. Also ask him about that Gurgaon restaurant."
}

created = requests.post(f"{base}/api/notes", json=payload).json()
print("POST /api/notes ->")
created


POST /api/notes ->


{'id': 31,
 'text': 'Met Rohan yesterday. He recommended Atomic Habits. Need to order it this weekend. Also ask him about that Gurgaon restaurant.',
 'created_at': '2026-09-08T13:40:57.125376+00:00'}

In [ ]:
notes = requests.get(f"{base}/api/notes").json()
print("GET /api/notes ->")
notes


GET /api/notes ->


[{'id': 1,
  'text': 'Met Rohan. Need to ask him about the Gurgaon restaurant.',
  'created_at': '2026-09-08T12:45:19.144155+00:00'},
 {'id': 2,
  'text': 'We are building a chatbot.',
  'created_at': '2026-09-08T12:46:37.967144+00:00'},
 {'id': 3,
  'text': 'Need to revise whatever I learnt.',
  'created_at': '2026-09-08T12:46:37.975958+00:00'},
 {'id': 4,
  'text': 'Confirm final designs with Maya by Friday morning.',
  'created_at': '2026-09-08T13:10:27.416394+00:00'},
 {'id': 5,
  'text': 'Pay electric bill before 6/1 to avoid late fee.',
  'created_at': '2026-09-08T13:10:27.426627+00:00'},
 {'id': 6,
  'text': 'Schedule dentist cleaning, try for next Wednesday.',
  'created_at': '2026-09-08T13:10:27.436164+00:00'},
 {'id': 7,
  'text': 'Buy replacement AirPods case and phone charger.',
  'created_at': '2026-09-08T13:10:27.444999+00:00'},
 {'id': 8,
  'text': 'Call Tom about Q2 partnership update ASAP.',
  'created_at': '2026-09-08T13:10:27.453600+00:00'},
 {'id': 9,
  'text': 'Res

## 7. OpenAI API demo: organize the whole inbox

This is the moment where the product becomes intelligent.

The important change is that we do **not** analyze one note in isolation.

We send the full saved inbox to the backend, and the LLM turns the messy set into:

- useful buckets,
- a priority view,
- concrete ToDos,
- links back to the source note IDs.

This is closer to the actual product idea: **capture first, organize later**.


In [ ]:
analysis = requests.post(f"{base}/api/analyze").json()

print("POST /api/analyze ->")
print(json.dumps(analysis, indent=2, ensure_ascii=False))


POST /api/analyze ->
{
  "buckets": [
    {
      "name": "Work",
      "notes": [
        {
          "id": 2,
          "title": "Building chatbot",
          "priority": "Low"
        },
        {
          "id": 4,
          "title": "Confirm final designs with Maya",
          "priority": "High"
        },
        {
          "id": 8,
          "title": "Call Tom about Q2 partnership update",
          "priority": "High"
        },
        {
          "id": 11,
          "title": "Draft and send 1-page investor update",
          "priority": "High"
        },
        {
          "id": 17,
          "title": "Send UX feedback to Ben",
          "priority": "High"
        },
        {
          "id": 20,
          "title": "Follow up with legal on NDA edits",
          "priority": "High"
        }
      ]
    },
    {
      "name": "Personal",
      "notes": [
        {
          "id": 1,
          "title": "Ask Rohan about Gurgaon restaurant",
          "priority": "Medium"
       

### Architecture Change



```text
Before: Client -> Backend -> Database

After:  Client -> Backend -> Database
                    |
                    v
                 OpenAI API
```

The backend first reads the **complete inbox** from SQLite and then sends a structured representation of those notes to the LLM.

Emphasize:

- The LLM sees the notes as context for this request. We did not retrain it.
- The backend owns the API key.
- The browser never sees the secret key.
- One LLM call can reason across many saved notes together.
- The prompt asks for machine-readable structure: buckets, ToDos, and priorities.
- LLM output is useful data, but it still needs validation and evaluation before you would trust it in a production product.


## 8. DevTools moment inside our own app

Open the app. Then:

1. Right-click -> Inspect.
2. Open Network tab.
3. Clear the tab.
4. Add a note.
5. Click the `/api/notes` request.
6. Show Payload and Response.
7. Add a few more notes, or use the optional bulk-population block.
8. Click **Analyze Entire Inbox with AI**.
9. Click the `/api/analyze` request and inspect its response.

Ask students:

- Which request wrote data?
- Why does `/api/analyze` not need the note text from the browser?
- Where does the backend get the notes it analyzes?
- Where is the database?
- Where is the API key?
- What would change if the client were a mobile app?


## 9. Closing reflection


> What clicked today that you did not expect?

Expected answers may include:

- Websites are just request/response flows.
- The backend is where business logic runs.
- The database is what remembers.
- The LLM is only one component in an AI product.
- AI can help both outside the product as a coding buddy and inside the product as a feature.
